# Dataset smoke test

Run one 32-token prompt from a text dataset through a single model using the analyzer pipeline. If needed, swap `DatasetPromptSource` for `RandomTokenPromptSource` without changing the rest of the code.

In [ ]:
import sys
import torch
import pandas as pd

sys.path.insert(0, '..')

from config import MODEL_NAME_SMALL, MODEL_NAME, OUTPUT_PATH
from core.analyzer import LightweightAttentionAnalyzer
from data.prompt_sources import DatasetPromptSource
from pipeline import run_analysis

if torch.backends.mps.is_available():
    device = "mps"
elif torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"

analyzer = LightweightAttentionAnalyzer(
    model_name=MODEL_NAME,
    device=device,
    local_files_only=False,
)

source = DatasetPromptSource(
    tokenizer=analyzer.tokenizer,
    dataset_name="wikitext",
    dataset_config="wikitext-103-raw-v1",
    split="train",
    target_tokens=32,
)





[Analyzer] Loading model 'Qwen/Qwen3-4B' on device 'mps'...


Loading weights: 100%|██████████| 398/398 [00:00<00:00, 1262.14it/s, Materializing param=model.norm.weight]                              


[Analyzer] Model loaded successfully. Using 36 layers.

[Analyzer] Processing prompt: ' Senjō no Valkyria 3 : Unrecorded Chronicles ( Japanese : 戦場のヴァルキュリア3 , lit . Va...'
[Analyzer] Tokenized to 32 tokens.
[Analyzer] Running forward pass...
[Analyzer] Analyzing 1 layers x 1 heads = 1 total heads.
[Analyzer]   Layer 0/35...
[Analyzer] Completed analysis. Extracted 1 head-level feature sets.
[Save] rows_added=1 total_rows=3


In [22]:
df = run_analysis(
    analyzer=analyzer,
    source=source,
    prompt_idx=0,
    layer_indices=[0],
    head_indices=[3, 4, 5],
    output_path=OUTPUT_PATH,
)


[Analyzer] Processing prompt: ' Senjō no Valkyria 3 : Unrecorded Chronicles ( Japanese : 戦場のヴァルキュリア3 , lit . Va...'
[Analyzer] Tokenized to 32 tokens.
[Analyzer] Running forward pass...
[Analyzer] Analyzing 1 layers x 3 heads = 3 total heads.
[Analyzer]   Layer 0/35...
[Analyzer] Completed analysis. Extracted 3 head-level feature sets.
[Save] rows_added=2 total_rows=5


In [23]:
df.head(5)

,model_name,layer_idx,head_idx,prompt_len,prompt_source,effective_rank_Wq,r95_Wq,effective_rank_Wk,r95_Wk,effective_rank_Wv,...,sink_mass_token_4,attention_entropy,attention_gini,max_attention_weight,attention_variance_per_query,query_key_sim_mean,attention_center_of_mass,effective_rank_A,r95_A,prompt_id
0,Qwen/Qwen2.5-0.5B-Instruct,0,0,32,wikitext_wikitext-103-raw-v1_train,36.674137,47.0,36.341274,46.0,57.515305,...,0.060452,59.401817,0.846919,1.0,0.007250,0.072131,0.810145,21.629070,23.0,wikitext_wikitext-103-raw-v1_train_doc0_32tok
1,Qwen/Qwen2.5-0.5B-Instruct,0,3,32,wikitext_wikitext-103-raw-v1_train,34.178291,47.0,36.341274,46.0,57.515305,...,0.111430,69.357117,0.776905,1.0,0.005758,0.148905,0.640179,17.817232,20.0,wikitext_wikitext-103-raw-v1_train_doc0_32tok
2,Qwen/Qwen3-4B,0,3,32,wikitext_wikitext-103-raw-v1_train,111.996078,111.0,122.341995,115.0,126.055634,...,0.015719,43.203468,0.904800,1.0,0.013325,0.273002,0.800958,23.837214,23.0,wikitext_wikitext-103-raw-v1_train_doc0_32tok
3,Qwen/Qwen3-4B,0,4,32,wikitext_wikitext-103-raw-v1_train,124.568443,117.0,123.263565,116.0,126.692337,...,0.007173,57.464630,0.853034,1.0,0.008167,-0.072526,0.721976,19.084234,20.0,wikitext_wikitext-103-raw-v1_train_doc0_32tok
4,Qwen/Qwen3-4B,0,5,32,wikitext_wikitext-103-raw-v1_train,121.009537,114.0,123.263565,116.0,126.692337,...,0.001009,26.018093,0.944773,1.0,0.020954,0.306907,0.884020,25.376560,23.0,wikitext_wikitext-103-raw-v1_train_doc0_32tok


In [24]:
dataframe = pd.read_parquet('data/attention_features.parquet')

In [25]:
dataframe.head()

,model_name,layer_idx,head_idx,prompt_len,prompt_source,effective_rank_Wq,r95_Wq,effective_rank_Wk,r95_Wk,effective_rank_Wv,...,sink_mass_token_4,attention_entropy,attention_gini,max_attention_weight,attention_variance_per_query,query_key_sim_mean,attention_center_of_mass,effective_rank_A,r95_A,prompt_id
0,Qwen/Qwen2.5-0.5B-Instruct,0,0,32,wikitext_wikitext-103-raw-v1_train,36.674137,47.0,36.341274,46.0,57.515305,...,0.060452,59.401817,0.846919,1.0,0.007250,0.072131,0.810145,21.629070,23.0,wikitext_wikitext-103-raw-v1_train_doc0_32tok
1,Qwen/Qwen2.5-0.5B-Instruct,0,3,32,wikitext_wikitext-103-raw-v1_train,34.178291,47.0,36.341274,46.0,57.515305,...,0.111430,69.357117,0.776905,1.0,0.005758,0.148905,0.640179,17.817232,20.0,wikitext_wikitext-103-raw-v1_train_doc0_32tok
2,Qwen/Qwen3-4B,0,3,32,wikitext_wikitext-103-raw-v1_train,111.996078,111.0,122.341995,115.0,126.055634,...,0.015719,43.203468,0.904800,1.0,0.013325,0.273002,0.800958,23.837214,23.0,wikitext_wikitext-103-raw-v1_train_doc0_32tok
3,Qwen/Qwen3-4B,0,4,32,wikitext_wikitext-103-raw-v1_train,124.568443,117.0,123.263565,116.0,126.692337,...,0.007173,57.464630,0.853034,1.0,0.008167,-0.072526,0.721976,19.084234,20.0,wikitext_wikitext-103-raw-v1_train_doc0_32tok
4,Qwen/Qwen3-4B,0,5,32,wikitext_wikitext-103-raw-v1_train,121.009537,114.0,123.263565,116.0,126.692337,...,0.001009,26.018093,0.944773,1.0,0.020954,0.306907,0.884020,25.376560,23.0,wikitext_wikitext-103-raw-v1_train_doc0_32tok
